# 04 · Smoke run QLoRA — Qwen2.5-VL-3B trên ViVQA

**Mục tiêu: KHÔNG phải để có adapter tốt.** Mục tiêu là trả lời ba câu hỏi bằng số:

| # | Câu hỏi | Đo bằng |
|---|---|---|
| 1 | Đường ray training có chạy được không (nạp 4-bit → LoRA → che nhãn → backward)? | Chạy hết 30 step, loss là số hữu hạn |
| 2 | Một step tốn bao nhiêu? | `sec_per_step` |
| 3 | `max_pixels` 512 hay 1280 khi train? | `peak_vram_gb` + `sec_per_step` của hai run |

Câu 3 chính là **dữ kiện còn thiếu** để chốt config ở W7. Ablation W6 đã đo chất lượng và chi phí *inference*; nhưng độ phân giải ảnh hưởng nặng hơn nhiều khi *train*, vì phải giữ activation cho backward.

**Dự đoán trước khi chạy (ghi ra để khỏi tự bào chữa sau):** run 1280 token nhiều khả năng **OOM trên T4 15GB**. Nếu OOM thật thì đó vẫn là kết quả hợp lệ — nghĩa là *trên T4, 1280 token không train được* → chọn 512 cho W6–W7 và đo lại trên L4.

**Điều kiện:** Runtime → Change runtime type → **T4 GPU**. Data ViVQA (train.json + ảnh COCO) đã có trên Drive.

## 1. Thiết lập môi trường

Mount Drive, clone/pull code, cài thư viện. So với notebook 01–03 có thêm **`peft`** (thư viện LoRA) và **`trl` không cần** — ta dùng `Trainer` chuẩn của `transformers`, ít phụ thuộc hơn.

*Kỳ vọng: commit ≥ `c9ab5f7`, GPU `Tesla T4`, `✅ Thiết lập xong`. ~2 phút.*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')

!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU -> đổi runtime sang T4')
print('✅ Thiết lập xong')

## 2. Kiểm tra trước khi chạy (rẻ, tránh mất công)

**Vấn đề:** nạp model mất ~4 phút. Nếu thiếu file train hoặc thiếu ảnh thì script sẽ chết **sau** 4 phút đó. Cell này kiểm tra trong vài giây, trước khi đụng tới GPU.

Kiểm ba thứ: `train.json` tồn tại, thư mục ảnh có ảnh, và selftest của script vẫn đạt trên đúng commit này.

*Kỳ vọng: in số mẫu train, số ảnh, và `SELFTEST OK`.*

In [ ]:
import json

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
TRAIN_JSON = f'{DRIVE_ROOT}/data/vivqa/train.json'
IMG_DIR    = f'{DRIVE_ROOT}/data/vivqa/images'

assert os.path.exists(TRAIN_JSON), f'Thiếu {TRAIN_JSON} -> chạy lại src/data/prepare_vivqa.py'
train = json.load(open(TRAIN_JSON, encoding='utf-8'))
print('train.json :', len(train), 'mẫu')
print('mẫu đầu    :', json.dumps(train[0], ensure_ascii=False)[:200])

assert os.path.isdir(IMG_DIR), f'Thiếu thư mục ảnh {IMG_DIR}'
n_img = len(os.listdir(IMG_DIR))
print('ảnh trên Drive:', n_img)

!python -m src.adaptation.train_qlora --selftest

## 3. Smoke run A — `max_pixels` 512 token

30 step trên 200 mẫu đầu, `compute_dtype = float32` (theo kết luận dtype probe ở notebook 03).

**Vì sao gọi bằng `!python` thay vì import vào notebook:** mỗi run là một tiến trình riêng, nên khi kết thúc **VRAM được trả lại sạch sẽ**. Nếu chạy hai run trong cùng kernel, run thứ hai gần như chắc chắn OOM vì bộ nhớ của run đầu chưa được giải phóng hết.

*Kỳ vọng: in `trainable ... (<1%)`, chạy hết 30 step, dòng `== XONG: ... s/step | VRAM đỉnh ... GB`. ~8–15 phút (gồm ~4 phút nạp model + chép ảnh lần đầu).*

In [ ]:
!python -u -m src.adaptation.train_qlora \
    --dataset vivqa \
    --run_name W06_smoke_qlora_qwen25vl_vivqa_mp512_seed42 \
    --limit 200 --max_steps 30 \
    --max_pixels 401408 --dtype float32

## 4. Smoke run B — `max_pixels` 1280 token

Giống hệt run A, **chỉ đổi `--max_pixels`**. Một biến, một kết luận.

⚠️ **Run này có thể OOM.** Nếu thấy `CUDA out of memory`, ĐỪNG coi là hỏng — đó là câu trả lời cho câu hỏi 3. Chụp lại dòng lỗi rồi chạy tiếp cell tổng hợp; cell đó xử lý được trường hợp thiếu một run.

*Kỳ vọng: hoặc chạy xong với `sec_per_step` cao hơn rõ rệt, hoặc OOM. Cả hai đều là kết quả dùng được.*

In [ ]:
!python -u -m src.adaptation.train_qlora \
    --dataset vivqa \
    --run_name W06_smoke_qlora_qwen25vl_vivqa_mp1280_seed42 \
    --limit 200 --max_steps 30 \
    --max_pixels 1003520 --dtype float32

## 5. So sánh chi phí train

Đọc `train_log.json` của hai run, in bảng chi phí và **kết luận theo tiêu chí cố định trước**:

| Tình huống | Kết luận |
|---|---|
| B không có log | Không tự kết luận; đọc log cell mục 4 (OOM hay bị ngắt) |
| B chạy được, chậm hơn > 1,8× **hoặc** VRAM > 13 GB | 512 — biên an toàn quá mỏng để chạy 8 ablation |
| B chạy được, chậm hơn ≤ 1,8× và VRAM ≤ 13 GB | Để ngỏ, quyết ở W7 trên L4 cùng dtype |

*Kỳ vọng: bảng 1–2 dòng + một dòng KẾT LUẬN, và file `smoke_summary_nb04.json`.*

Bản tổng hợp chuẩn (4 run, có ước tính epoch) nằm ở `smoke_summary.json` trong repo; cell này chỉ ghi bản nháp riêng, không ghi đè bản chuẩn.

In [ ]:
import json, os
import pandas as pd

EXP = f'{DRIVE_ROOT}/experiments'
RUNS = {
    'mp512':  'W06_smoke_qlora_qwen25vl_vivqa_mp512_seed42',
    'mp1280': 'W06_smoke_qlora_qwen25vl_vivqa_mp1280_seed42',
}

rows, metas = [], {}
for tag, run in RUNS.items():
    path = f'{EXP}/{run}/train_log.json'
    if not os.path.exists(path):
        print(f'{tag}: KHÔNG có train_log.json -> chưa xong (OOM, bị ngắt, hoặc chưa chạy) — xem log của cell tương ứng')
        rows.append({'tag': tag, 'status': 'no_log'})
        continue
    m = json.load(open(path, encoding='utf-8'))['meta']
    metas[tag] = m
    rows.append({'tag': tag, 'status': 'ok', 'max_pixels': m['max_pixels'],
                 'dtype': m['torch_dtype'],
                 'sec_per_step': m['cost']['sec_per_step'],
                 'peak_vram_gb': m['cost']['peak_vram_gb'],
                 'final_loss': m['cost']['final_loss'],
                 'trainable_pct': m['trainable_pct'],
                 'steps': m['cost']['steps']})

df = pd.DataFrame(rows)
display(df)

a, b = metas.get('mp512'), metas.get('mp1280')
if a:
    print(f"\nPEFT-only: trainable {a['trainable_params']:,} / {a['total_params']:,} "
          f"= {a['trainable_pct']}%  (bằng chứng không full fine-tuning)")

if a and b:
    slow = b['cost']['sec_per_step'] / max(a['cost']['sec_per_step'], 1e-9)
    vram = b['cost']['peak_vram_gb']
    print(f"1280 so với 512: chậm {slow:.2f}×, VRAM đỉnh {vram:.2f} GB")
    if slow > 1.8 or vram > 13:
        verdict = 'CHỐT 512 — biên an toàn quá mỏng cho 8 ablation ở W8–W12.'
    else:
        verdict = 'ĐỂ NGỎ — 1280 khả thi trên T4; quyết ở W7 trên L4 cùng dtype.'
elif a and not b:
    verdict = 'THIẾU RUN 1280 — không kết luận được về 1280; kiểm tra log cell mục 4 (OOM hay bị ngắt).'
else:
    verdict = 'THIẾU RUN 512 — không kết luận được; kiểm tra log cell mục 3.'

print('\n' + '='*70)
print('KẾT LUẬN:', verdict)
print('='*70)

OUT = f'{EXP}/W06_smoke_qlora_seed42'
os.makedirs(OUT, exist_ok=True)
with open(f'{OUT}/smoke_summary_nb04.json', 'w', encoding='utf-8') as fp:
    json.dump({'runs': rows, 'verdict': verdict}, fp, ensure_ascii=False, indent=2)
print('Ghi ->', f'{OUT}/smoke_summary_nb04.json')

## 6. Soi nhanh loss (kiểm tra đường ray, không phải kiểm tra chất lượng)

30 step là **quá ngắn** để nói adapter tốt hay xấu. Nhưng loss vẫn cho biết đường ray có hỏng không:

| Dấu hiệu | Nghĩa là |
|---|---|
| Loss là `nan` hoặc `inf` | Tràn số — vấn đề dtype quay lại ở mức nghiêm trọng hơn |
| Loss đứng im hoàn toàn | Nghi che nhãn sai hoặc LoRA không được gắn |
| Loss dao động nhưng có xu hướng giảm | Bình thường với 30 step |

*Kỳ vọng: vài dòng loss, đều là số hữu hạn.*

In [ ]:
path = f"{EXP}/{RUNS['mp512']}/train_log.json"
if os.path.exists(path):
    hist = json.load(open(path, encoding='utf-8'))['log_history']
    for h in hist:
        if 'loss' in h:
            print(f"step {h.get('step'):>3} | loss {h['loss']:.4f} | lr {h.get('learning_rate', 0):.2e}")
else:
    print('Chưa có train_log.json cho run 512.')

## 7. Bước tiếp theo

Dán cho Claude: **bảng mục 5**, dòng KẾT LUẬN, dòng PEFT-only, và **vài dòng loss ở mục 6**.

Nếu run B báo `CUDA out of memory`, dán thêm **dòng lỗi cuối cùng** — nó cho biết thiếu bao nhiêu GB, là số hữu ích để ước tính có vừa L4 22,5 GB hay không.

Sau đó Claude sẽ: nghiệm thu, sinh prompt Cursor chép kết quả + commit, và chuyển sang các việc còn lại của W6 (prompt optimization, gán `question_type`, Methodology R4, dàn ý Review 2).